##### Q5. Import Datasets from Different Sources
First, install the required Python libraries.

pip install pandas numpy matplotlib seaborn requests sqlalchemy ydata-profiling scikit-learn

1. Import CSV File Using Pandas

In [5]:
import pandas as pd

df_csv = pd.read_csv(r"C:\Users\bharg\Desktop\pr dpfe\customer_churn_dataset.csv", sep=';')

print(df_csv.head())

  CustomerID,Age,Gender,Income,Purchases,TotalSpending,DaysSinceLastPurchase,Churn
0       CUST0001,56.0,Male,44375.0,10,19092.31,102,0                              
1        CUST0002,69.0,Female,31964.0,5,9212.02,75,0                              
2      CUST0003,46.0,Female,128429.0,2,3845.73,161,1                              
3       CUST0004,32.0,Male,82215.0,11,24998.63,151,1                              
4        CUST0005,60.0,Male,84042.0,5,10555.11,143,1                              


2. Parse JSON File

In [9]:
import json
import pandas as pd

with open("customers.json", "r") as file:
    data = json.load(file)

# record_path points to the nested list
# meta keeps the parent customer-level info for each row
df = pd.json_normalize(
    data,
    record_path=['orders'],
    meta=['customer_id', 'name']
)

print(df)

   order_id  amount customer_id   name
0       101      50           1  Alice
1       102     120           1  Alice


3. Connect to SQL Database and Fetch Records

In [11]:
import sqlite3
import pandas as pd

# Connect to SQLite database (creates 'customer.db' if it doesn't exist)
conn = sqlite3.connect("customer.db")

# Optional: Create a sample table and insert data if it doesn't exist yet
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS customers (
        id INTEGER PRIMARY KEY,
        name TEXT,
        city TEXT
    )
""")
# Insert dummy data if table is empty
cursor.execute("INSERT OR IGNORE INTO customers (id, name, city) VALUES (1, 'Alice', 'New York')")
conn.commit()

# Query the database
query = "SELECT * FROM customers"
df_sql = pd.read_sql_query(query, conn)

print(df_sql.head())

# Always close the connection when done
conn.close()

   id   name      city
0   1  Alice  New York


### 4. Fetch Data from an API

Example: DummyJSON API.

In [12]:
import requests
import pandas as pd

url = "https://dummyjson.com/users"

response = requests.get(url)
data = response.json()

df_api = pd.json_normalize(data["users"])

print(df_api.head())

   id firstName  lastName maidenName  age  gender  \
0   1     Emily   Johnson      Smith   29  female   
1   2   Michael  Williams              36    male   
2   3    Sophia     Brown              43  female   
3   4     James     Davis              46    male   
4   5      Emma    Miller    Johnson   31  female   

                              email             phone  username      password  \
0     emily.johnson@x.dummyjson.com  +81 965-431-3024    emilys    emilyspass   
1  michael.williams@x.dummyjson.com  +49 258-627-6644  michaelw  michaelwpass   
2      sophia.brown@x.dummyjson.com  +81 210-652-2785   sophiab   sophiabpass   
3       james.davis@x.dummyjson.com  +49 614-958-9364    jamesd    jamesdpass   
4       emma.miller@x.dummyjson.com  +91 759-776-1614     emmaj     emmajpass   

   ... company.address.city company.address.state company.address.stateCode  \
0  ...        San Francisco             Wisconsin                        WI   
1  ...          Los Angeles         

## Part C: Data Understanding and Cleaning

Q6. Perform Initial Exploration
1. Display First Five Records

In [16]:
print(df.head())

   order_id  amount customer_id   name
0       101      50           1  Alice
1       102     120           1  Alice


2. Display Dataset Information

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   order_id     2 non-null      int64 
 1   amount       2 non-null      int64 
 2   customer_id  2 non-null      object
 3   name         2 non-null      str   
dtypes: int64(2), object(1), str(1)
memory usage: 206.0+ bytes


3. Display Statistical Summary

In [18]:
print(df.describe())

         order_id      amount
count    2.000000    2.000000
mean   101.500000   85.000000
std      0.707107   49.497475
min    101.000000   50.000000
25%    101.250000   67.500000
50%    101.500000   85.000000
75%    101.750000  102.500000
max    102.000000  120.000000


4. Check Missing Values
   

In [19]:
print(df.isnull().sum())

order_id       0
amount         0
customer_id    0
name           0
dtype: int64


In [ ]:
5. Check Duplicate Records

In [20]:
print("Duplicates:", df.duplicated().sum())

Duplicates: 0


In [ ]:
6. Display Dataset Shape

In [21]:
print("Rows and Columns:", df.shape)

Rows and Columns: (2, 4)


### Q7. Apply Data Cleaning

1. Handle Missing Values

In [25]:
# Test if the code runs with dummy data containing non-breaking spaces or if there's any syntax error
import pandas as pd
import numpy as np

# Recreating the exact text with potential non-breaking spaces
code_str = """
df = pd.DataFrame({
    'Age': [25, np.nan, 30],
    'Income': [50000, 60000, np.nan],
    'Gender': ['M', 'F', np.nan]
})

df["Age"] = df["Age"].fillna(df["Age"].median())

df["Income"] = df["Income"].fillna(
    df["Income"].median()
)

# Fill missing categorical values with mode

df["Gender"] = df["Gender"].fillna(
    df["Gender"].mode()[0]
)
"""
try:
    exec(code_str)
    print("Execution successful!")
except Exception as e:
    print("Error:", type(e), e)

Execution successful!


In [27]:
import pandas as pd

# Fill missing numerical values with the median
df["Age"] = df["Age"].fillna(df["Age"].median())

df["Income"] = df["Income"].fillna(df["Income"].median())

# Fill missing categorical values with the mode (using [0] in case of multiple modes)
df["Gender"] = df["Gender"].fillna(df["Gender"].mode()[0])

2. Remove Duplicate Records

In [30]:
df = df.drop_duplicates()

3. Correct Inconsistent Data Types

In [35]:
# Counts the number of NaN/NaT values per column
print(df.isna().sum())

Age       0
Income    0
Gender    0
dtype: int64


4. Drop Irrelevant Columns

In [40]:
df = df.drop(
    columns=["Address", "Phone"],
    errors="ignore"
)

In [ ]:
5. Final Check

In [41]:
print(df.isnull().sum())
print(df.duplicated().sum())
print(df.info())

Age       0
Income    0
Gender    0
dtype: int64
0
<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Age     3 non-null      float64
 1   Income  3 non-null      float64
 2   Gender  3 non-null      str    
dtypes: float64(2), str(1)
memory usage: 207.0 bytes
None


### Part D: Exploratory Data Analysis (EDA)

In [4]:
import pandas as pd
from pathlib import Path

# Example usage with dummy data:
data = {
    "Age": [25, 32, 47, 51, 22, 38, 45, 29],
    "Income": [50000, 65000, 120000, 85000, 42000, 78000, 95000, 48000],
    "Purchases": [3, 5, 12, 8, 2, 7, 10, 4],
    "PurchaseAmount": [150.5, 300.2, 1200.0, 750.0, 90.0, 450.0, 890.0, 210.0],
    "Gender": ["M", "F", "F", "M", "F", "M", "F", "M"],
    "Churn": [0, 1, 0, 0, 1, 0, 1, 0]
}

df = pd.DataFrame(data)

# Call the function
create_eda_plots(df, output_dir="./eda_outputs")
print("Plots generated successfully in ./eda_outputs!")

Plots generated successfully in ./eda_outputs!


In [5]:
from pathlib import Path
import pandas as pd

def generate_profile_report(df: pd.DataFrame, output_file: str | Path):
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    # Handle empty dataframe edge case safely
    row_count = len(df)
    col_count = len(df.columns)
    
    if row_count == 0:
        print("Warning: The dataset is empty. Profiling report will show zero rows.")

    # Missing values summary
    missing = df.isna().sum().to_frame("Missing Values")
    missing["Missing %"] = (missing["Missing Values"] / row_count * 100).round(2) if row_count > 0 else 0.0

    # Descriptive statistics
    stats = df.describe(include="all").T.fillna("") if row_count > 0 else pd.DataFrame()

    # Correlation matrix (only if numeric columns exist)
    numeric_df = df.select_dtypes(include="number")
    if not numeric_df.empty and row_count > 1:
        corr = numeric_df.corr().round(3)
        corr_html = corr.to_html()
    else:
        corr_html = "<p>No numeric columns available for correlation analysis.</p>"

    # Automated warnings
    warnings = []
    dup_count = df.duplicated().sum()
    if dup_count > 0:
        warnings.append(f"{dup_count} duplicate rows detected.")
        
    for col in df.columns:
        if row_count > 0:
            pct = df[col].isna().mean() * 100
            if pct > 0:
                warnings.append(f"{col}: {pct:.2f}% missing values.")
                
    for col in numeric_df.columns:
        if df[col].nunique(dropna=True) <= 1:
            warnings.append(f"{col}: constant or near-constant numeric column.")
            
    if "CustomerID" in df.columns and df["CustomerID"].duplicated().any():
        warnings.append("CustomerID is not unique.")

    warning_html = "".join(f"<li>{w}</li>" for w in warnings) or "<li>No major automated warnings detected.</li>"

    # HTML Template Construction
    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="utf-8">
        <title>Customer Data Profiling Report</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 32px; color: #333; line-height: 1.5; }}
            h1, h2 {{ color: #183153; }}
            table {{ border-collapse: collapse; width: 100%; margin-bottom: 24px; font-size: 13px; }}
            th, td {{ border: 1px solid #ddd; padding: 8px 12px; text-align: left; }}
            th {{ background-color: #f4f6f9; color: #183153; }}
            tr:nth-child(even) {{ background-color: #fafbfc; }}
            .note {{ background: #eef5ff; padding: 12px 16px; border-left: 4px solid #183153; margin-bottom: 20px; }}
        </style>
    </head>
    <body>
        <h1>Customer Data Profiling Report</h1>
        <div class="note">Generated automatically from the raw combined customer dataset.</div>
        
        <h2>Dataset Overview</h2>
        <p><strong>Rows:</strong> {row_count} | <strong>Columns:</strong> {col_count} | <strong>Duplicate Rows:</strong> {dup_count}</p>
        
        <h2>Missing Values</h2>
        {missing.to_html()}
        
        <h2>Descriptive Statistics</h2>
        {stats.to_html() if not stats.empty else "<p>No data available.</p>"}
        
        <h2>Correlation Matrix</h2>
        {corr_html}
        
        <h2>Potential Data Quality Warnings</h2>
        <ul>{warning_html}</ul>
    </body>
    </html>
    """
    
    output_file.write_text(html, encoding="utf-8")
    print(f"Profiling report successfully saved: {output_file}")

if __name__ == "__main__":
    try:
        from data_acquisition import acquire_all_data
        raw = acquire_all_data()
        generate_profile_report(raw, "outputs/data_profiling_report.html")
    except ImportError:
        print("Note: 'data_acquisition' module not found. Ensure it is in your path when running.")

Note: 'data_acquisition' module not found. Ensure it is in your path when running.
